In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.chdir('/content/drive/MyDrive/credit-risk-assessment-system')

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [3]:
X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = pd.read_csv('data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('data/processed/y_test.csv').squeeze()

print(X_train.shape, X_test.shape)

(246008, 180) (61503, 180)


In [4]:
raw_df = pd.read_csv('data/raw/application_train.csv')
X_raw = raw_df.drop(columns=['TARGET'])
y_raw = raw_df['TARGET']

from sklearn.model_selection import train_test_split
X_train_raw, X_test_raw, _, _ = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

X_train['SK_ID_CURR'] = X_train_raw['SK_ID_CURR'].values
X_test['SK_ID_CURR'] = X_test_raw['SK_ID_CURR'].values

print(X_train['SK_ID_CURR'].nunique(), X_test['SK_ID_CURR'].nunique())

246008 61503


Bureau-based features


In [5]:
bureau = pd.read_csv('data/raw/bureau.csv')

bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_COUNT=('SK_ID_CURR', 'size'),
    BUREAU_MAX_OVERDUE=('AMT_CREDIT_SUM_OVERDUE', 'max'),
    BUREAU_HAS_OVERDUE=('AMT_CREDIT_SUM_OVERDUE', lambda x: (x > 0).any())
).reset_index()

print(bureau_agg.shape)
bureau_agg.head()

(305811, 4)


,SK_ID_CURR,BUREAU_COUNT,BUREAU_MAX_OVERDUE,BUREAU_HAS_OVERDUE
0,100001,7,0.0,False
1,100002,8,0.0,False
2,100003,4,0.0,False
3,100004,2,0.0,False
4,100005,3,0.0,False


Previous-application-based feature

In [6]:
prev_app = pd.read_csv('data/raw/previous_application.csv')

prev_agg = prev_app.groupby('SK_ID_CURR').agg(
    PREV_APP_COUNT=('SK_ID_CURR', 'size'),
    PREV_WAS_REFUSED=('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').any())
).reset_index()

print(prev_agg.shape)
prev_agg.head()

(338857, 3)


,SK_ID_CURR,PREV_APP_COUNT,PREV_WAS_REFUSED
0,100001,1,False
1,100002,1,False
2,100003,3,False
3,100004,1,False
4,100005,2,False


Installment-lateness feature

In [7]:
installments = pd.read_csv('data/raw/installments_payments.csv')
installments['DAYS_LATE'] = (
    installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']
).clip(lower=0)

installment_agg = installments.groupby('SK_ID_CURR').agg(
    AVG_DAYS_LATE=('DAYS_LATE', 'mean'),
    MAX_DAYS_LATE=('DAYS_LATE', 'max')
).reset_index()

print(installment_agg.shape)
installment_agg.head()

(339587, 3)


,SK_ID_CURR,AVG_DAYS_LATE,MAX_DAYS_LATE
0,100001,1.571429,11.0
1,100002,0.000000,0.0
2,100003,0.000000,0.0
3,100004,0.000000,0.0
4,100005,0.111111,1.0


In [8]:
def merge_features(X, bureau_agg, prev_agg, installment_agg):
    X = X.merge(bureau_agg, on='SK_ID_CURR', how='left')
    X = X.merge(prev_agg, on='SK_ID_CURR', how='left')
    X = X.merge(installment_agg, on='SK_ID_CURR', how='left')
    return X

X_train_fe = merge_features(X_train, bureau_agg, prev_agg, installment_agg)
X_test_fe = merge_features(X_test, bureau_agg, prev_agg, installment_agg)

print(X_train_fe.shape, X_test_fe.shape)

(246008, 187) (61503, 187)


In [9]:
new_cols = ['BUREAU_COUNT', 'BUREAU_MAX_OVERDUE', 'BUREAU_HAS_OVERDUE',
            'PREV_APP_COUNT', 'PREV_WAS_REFUSED',
            'AVG_DAYS_LATE', 'MAX_DAYS_LATE']

print(X_train_fe[new_cols].isnull().sum())

BUREAU_COUNT          35244
BUREAU_MAX_OVERDUE    35244
BUREAU_HAS_OVERDUE    35244
PREV_APP_COUNT        13174
PREV_WAS_REFUSED      13174
AVG_DAYS_LATE         12691
MAX_DAYS_LATE         12691
dtype: int64


Step 1 — capture "no history" as its own flag, before filling

In [10]:
X_train_fe['HAS_BUREAU_HISTORY'] = X_train_fe['BUREAU_COUNT'].notna()
X_test_fe['HAS_BUREAU_HISTORY'] = X_test_fe['BUREAU_COUNT'].notna()

X_train_fe['HAS_PREV_APP'] = X_train_fe['PREV_APP_COUNT'].notna()
X_test_fe['HAS_PREV_APP'] = X_test_fe['PREV_APP_COUNT'].notna()

X_train_fe['HAS_INSTALLMENT_HISTORY'] = X_train_fe['AVG_DAYS_LATE'].notna()
X_test_fe['HAS_INSTALLMENT_HISTORY'] = X_test_fe['AVG_DAYS_LATE'].notna()

Step 2 — now fill the actual missing values with 0 (or False), since "no history" logically means "count of zero"

In [11]:
fill_zero_cols = ['BUREAU_COUNT', 'BUREAU_MAX_OVERDUE', 'PREV_APP_COUNT',
                   'AVG_DAYS_LATE', 'MAX_DAYS_LATE']
fill_false_cols = ['BUREAU_HAS_OVERDUE', 'PREV_WAS_REFUSED']

for col in fill_zero_cols:
    X_train_fe[col] = X_train_fe[col].fillna(0)
    X_test_fe[col] = X_test_fe[col].fillna(0)

for col in fill_false_cols:
    X_train_fe[col] = X_train_fe[col].fillna(False).astype(int)
    X_test_fe[col] = X_test_fe[col].fillna(False).astype(int)

print(X_train_fe[new_cols].isnull().sum().sum())

0


/tmp/ipykernel_34673/1551498328.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train_fe[col] = X_train_fe[col].fillna(False).astype(int)
/tmp/ipykernel_34673/1551498328.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test_fe[col] = X_test_fe[col].fillna(False).astype(int)
/tmp/ipykernel_34673/1551498328.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.

Now the ratio features from your original EDA — credit-to-income and annuity-to-income:

In [12]:
X_train_fe['CREDIT_INCOME_RATIO'] = X_train_fe['AMT_CREDIT'] / X_train_fe['AMT_INCOME_TOTAL']
X_test_fe['CREDIT_INCOME_RATIO'] = X_test_fe['AMT_CREDIT'] / X_test_fe['AMT_INCOME_TOTAL']

X_train_fe['ANNUITY_INCOME_RATIO'] = X_train_fe['AMT_ANNUITY'] / X_train_fe['AMT_INCOME_TOTAL']
X_test_fe['ANNUITY_INCOME_RATIO'] = X_test_fe['AMT_ANNUITY'] / X_test_fe['AMT_INCOME_TOTAL']

print(X_train_fe[['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO']].describe())

       CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO
count        246008.000000         246008.000000
mean              1.248034              1.503777
std              34.502605             32.233118
min           -6377.511028          -5857.541958
25%              -1.438008             -0.936605
50%               1.704029              1.651965
75%               4.155765              4.095435
max            2804.902324           3129.829901


Fix: pull the original, unscaled dollar values from the raw split (same trick as recovering SK_ID_CURR)

In [20]:
X_train_fe['AMT_CREDIT_RAW'] = X_train_raw['AMT_CREDIT'].values
X_train_fe['AMT_INCOME_TOTAL_RAW'] = X_train_raw['AMT_INCOME_TOTAL'].values
X_train_fe['AMT_ANNUITY_RAW'] = X_train_raw['AMT_ANNUITY'].values

X_test_fe['AMT_CREDIT_RAW'] = X_test_raw['AMT_CREDIT'].values
X_test_fe['AMT_INCOME_TOTAL_RAW'] = X_test_raw['AMT_INCOME_TOTAL'].values
X_test_fe['AMT_ANNUITY_RAW'] = X_test_raw['AMT_ANNUITY'].values

X_train_fe['CREDIT_INCOME_RATIO'] = X_train_fe['AMT_CREDIT_RAW'] / X_train_fe['AMT_INCOME_TOTAL_RAW']
X_test_fe['CREDIT_INCOME_RATIO'] = X_test_fe['AMT_CREDIT_RAW'] / X_test_fe['AMT_INCOME_TOTAL_RAW']

X_train_fe['ANNUITY_INCOME_RATIO'] = X_train_fe['AMT_ANNUITY_RAW'] / X_train_fe['AMT_INCOME_TOTAL_RAW']
X_test_fe['ANNUITY_INCOME_RATIO'] = X_test_fe['AMT_ANNUITY_RAW'] / X_test_fe['AMT_INCOME_TOTAL_RAW']

print(X_train_fe[['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO']].describe())

       CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO
count        246008.000000         245998.000000
mean              3.959396              0.180911
std               2.687597              0.094571
min               0.004808              0.000224
25%               2.018667              0.114583
50%               3.268948              0.162833
75%               5.168514              0.229000
max              49.227200              1.570600


In [14]:
X_train_fe = X_train_fe.drop(columns=['AMT_CREDIT_RAW', 'AMT_INCOME_TOTAL_RAW', 'AMT_ANNUITY_RAW'])
X_test_fe = X_test_fe.drop(columns=['AMT_CREDIT_RAW', 'AMT_INCOME_TOTAL_RAW', 'AMT_ANNUITY_RAW'])

In [21]:
print(X_train_fe['AMT_CREDIT_RAW'].describe())

count    2.460080e+05
mean     5.993382e+05
std      4.027258e+05
min      4.500000e+04
25%      2.700000e+05
50%      5.147775e+05
75%      8.086500e+05
max      4.050000e+06
Name: AMT_CREDIT_RAW, dtype: float64


In [22]:
median_annuity_ratio = X_train_fe['ANNUITY_INCOME_RATIO'].median()

X_train_fe['ANNUITY_INCOME_RATIO'] = X_train_fe['ANNUITY_INCOME_RATIO'].fillna(median_annuity_ratio)
X_test_fe['ANNUITY_INCOME_RATIO'] = X_test_fe['ANNUITY_INCOME_RATIO'].fillna(median_annuity_ratio)

print(X_train_fe['ANNUITY_INCOME_RATIO'].isnull().sum())
print(X_test_fe['ANNUITY_INCOME_RATIO'].isnull().sum())

0
0


In [23]:
X_train_fe.to_csv('data/processed/X_train_fe.csv', index=False)
X_test_fe.to_csv('data/processed/X_test_fe.csv', index=False)

print(X_train_fe.shape, X_test_fe.shape)

(246008, 195) (61503, 195)
